# Reality Demand Aggregation — Norte Amazonia Bolivia

This notebook builds the `Demands.csv` EnergyScope input file for each cluster, under the **reality** scenario.

Three demand sources are combined:
- **Source A** — grid-connected municipalities: real consumption measured by AETN, broken down by sector and end-use
- **Source B** — off-grid households (9,325 HH): RAMP reality simulation outputs
- **Source C** — non-electrified households (11,389 HH): zero electrical demand

On top of electricity, **cooking** useful energy is added for all households not using electric stoves (from census data).

Output: `output_energyscope/C{k}/Demands.csv` for each cluster k.

## 0. Control — `Layers_in_out.csv` vs EnergyScope reference

Before building `Demands.csv`, check that the local `Layers_in_out.csv` (used below to convert RAMP/AETN electrical energy into useful service energy) matches the reference file used by the EnergyScope model itself, in `EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/`. A silent divergence here would corrupt every efficiency coefficient computed downstream.

In [1]:
import pandas as pd

LIO_LOCAL_PATH = "../data/Layers_in_out.csv"
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv"

lio_local = pd.read_csv(LIO_LOCAL_PATH, sep=";", header=0, index_col=0)
lio_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []

only_local_rows = sorted(set(lio_local.index) - set(lio_reference.index))
only_reference_rows = sorted(set(lio_reference.index) - set(lio_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_local.columns) - set(lio_reference.columns))
only_reference_cols = sorted(set(lio_reference.columns) - set(lio_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_local.index) & set(lio_reference.index))
common_cols = sorted(set(lio_local.columns) & set(lio_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_local.loc[tech, common_cols].equals(lio_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )

print(f"OK — Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")

OK — Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv)


In [2]:
import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "output_energyscope"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

MUNI_TO_CLUSTER = {m: k for k, munis in CLUSTERS.items() for m in munis}

# Source A CSV uses display names with spaces; canonical names use underscores
SOURCE_A_TO_RAMP = {
    "Bella Flor":           "Bella_Flor",
    "Bolpebra":             "Bolpebra",
    "Cobija":               "Cobija",
    "El Sena":              "Sena",
    "Exaltación":           "Exaltación",
    "Filadelfia":           "Filadelfia",
    "Guayaramerín":         "Guayaramerín",
    "Ingavi":               "Ingavi",
    "Ixiamas":              "Ixiamas",
    "Nueva Esperanza":      "Nueva_Esperanza",
    "Porvenir":             "Porvenir",
    "Puerto Gonzalo Moreno":"Puerto_Gonzalo_Moreno",
    "Puerto Rico":          "Puerto_Rico",
    "Reyes":                "Reyes",
    "Riberalta":            "Riberalta",
    "San Lorenzo":          "San_Lorenzo",
    "San Pedro":            "San_Pedro",
    "Santa Rosa":           "Santa_Rosa_Beni",
    "Santa Rosa del Abuná": "Santa_Rosa_Pando",
    "Santos Mercado":       "Santos_Mercado",
    "Villa Nueva":          "Villa_Nueva",
}

# Census file has two "Santa Rosa" — disambiguated by (name, department)
CSVFINAL_TO_RAMP = {
    ("Ixiamas",               "La Paz"): "Ixiamas",
    ("Riberalta",             "Beni"):   "Riberalta",
    ("Guayaramerín",          "Beni"):   "Guayaramerín",
    ("Reyes",                 "Beni"):   "Reyes",
    ("Santa Rosa",            "Beni"):   "Santa_Rosa_Beni",
    ("Exaltación",            "Beni"):   "Exaltación",
    ("Cobija",                "Pando"):  "Cobija",
    ("Porvenir",              "Pando"):  "Porvenir",
    ("Bolpebra",              "Pando"):  "Bolpebra",
    ("Bella Flor",            "Pando"):  "Bella_Flor",
    ("Puerto Rico",           "Pando"):  "Puerto_Rico",
    ("San Pedro",             "Pando"):  "San_Pedro",
    ("Filadelfia",            "Pando"):  "Filadelfia",
    ("Puerto Gonzalo Moreno", "Pando"):  "Puerto_Gonzalo_Moreno",
    ("San Lorenzo",           "Pando"):  "San_Lorenzo",
    ("Sena",                  "Pando"):  "Sena",
    ("Santa Rosa",            "Pando"):  "Santa_Rosa_Pando",
    ("Ingavi",                "Pando"):  "Ingavi",
    ("Nueva Esperanza",       "Pando"):  "Nueva_Esperanza",
    ("Villa Nueva",           "Pando"):  "Villa_Nueva",
    ("Santos Mercado",        "Pando"):  "Santos_Mercado",
}

# RAMP column → (EnergyScope sector, end-use layer)
# Reality uses small_school instead of big_school; health_center and public_lighting
# are grid-only and already captured in Source A
MAPPING_RAMP_REALITY = {
    "sufficiency_illumination":            ("HOUSEHOLDS", "LIGHTING_R_C"),
    "sufficiency_ICT":                     ("HOUSEHOLDS", "ELECTRICITY"),
    "sufficiency_cold_storage":            ("HOUSEHOLDS", "FOOD_PRESERVATION"),
    "sufficiency_thermal_comfort":         ("HOUSEHOLDS", "SPACE_COOLING"),
    "small_school_illumination":           ("SERVICES",   "LIGHTING_R_C"),
    "small_school_ICT":                    ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_illumination": ("SERVICES",   "LIGHTING_R_C"),
    "entertainment_business_ICT":          ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_cold_storage": ("SERVICES",   "FOOD_PRESERVATION"),
    "rice_processing_rice_processing":     ("INDUSTRY",   "MECHANICAL_ENERGY_IND"),
    "restaurant_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "restaurant_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_kitchen":                  ("SERVICES",   "COOKING"),
    "store_illumination":                  ("SERVICES",   "LIGHTING_R_C"),
    "store_ICT":                           ("SERVICES",   "ELECTRICITY"),
    "store_cold_storage":                  ("SERVICES",   "FOOD_PRESERVATION"),
    "workshop_illumination":               ("SERVICES",   "LIGHTING_R_C"),
    "workshop_ICT":                        ("SERVICES",   "ELECTRICITY"),
    "workshop_machinery":                  ("SERVICES",   "MECHANICAL_ENERGY_COMM"),
}

## 1. Efficiency coefficients from `Layers_in_out`

RAMP outputs watts of **electricity**; Source A reports **MWh of final electrical energy**. EnergyScope demands are expressed in **useful service energy** (the actual service delivered).

For each end-use layer we look up the reference electric technology in `Layers_in_out.csv` and read its `|ELECTRICITY|` input coefficient:

$$\text{GWh}_{\text{useful}} = \frac{\text{GWh}_{\text{electrical}}}{|\text{coeff}|}$$

Layers mapped to `None` keep `coeff = 1.0` (electricity is already the service unit).

In [3]:
LAYER_TO_TECH = {
    'ELECTRICITY':                   None,
    'LIGHTING_R_C':                  'LED_BULB',
    'LIGHTING_P':                    'LED_LIGHT',
    'HEAT_HIGH_T':                   'IND_DIRECT_ELEC',
    'HEAT_LOW_T_SH':                 'DEC_DIRECT_ELEC',
    'HEAT_LOW_T_HW':                 'DEC_DIRECT_ELEC',
    'COOKING':                       'STOVE_ELEC',
    'PROCESS_COOLING':               'IND_ELEC_COLD',
    'SPACE_COOLING':                 'DEC_ELEC_COLD',
    'FOOD_PRESERVATION':             'REFRIGERATOR_EL',
    'MECHANICAL_ENERGY_COMM':        'COMM_MACHINERY_EL',
    'MECHANICAL_ENERGY_IND':         'IND_MACHINERY_EL',
    'MECHANICAL_ENERGY_MOV_AGR':     'TRACTOR_EL',
    'MECHANICAL_ENERGY_FIX_AGR':     'AGR_MACHINERY_EL',
    'MECHANICAL_ENERGY_MIN':         'MIN_MACHINERY_EL',
    'MECHANICAL_ENERGY_FISH_OTHERS': 'FISH_MACHINERY_EL',
    'NON_ENERGY':                    None,
}

MOBILITY_LAYERS = {'MOBILITY_PASSENGER', 'MOBILITY_FREIGHT', 'AVIATION_LONG_HAUL', 'SHIPPING'}

lio = pd.read_csv("../data/Layers_in_out.csv", sep=";", header=0, index_col=0)

LAYER_TO_COEFF = {}
for layer, tech in LAYER_TO_TECH.items():
    if tech is None:
        LAYER_TO_COEFF[layer] = 1.0
    else:
        LAYER_TO_COEFF[layer] = abs(float(lio.loc[tech, "ELECTRICITY"]))

print("Layer → |ELECTRICITY coefficient|:")
for layer, coeff in LAYER_TO_COEFF.items():
    print(f"  {layer:<35} {coeff:.6f}")

Layer → |ELECTRICITY coefficient|:
  ELECTRICITY                         1.000000
  LIGHTING_R_C                        2.941176
  LIGHTING_P                          2.941176
  HEAT_HIGH_T                         1.000000
  HEAT_LOW_T_SH                       1.000000
  HEAT_LOW_T_HW                       1.000000
  COOKING                             1.000000
  PROCESS_COOLING                     0.496500
  SPACE_COOLING                       0.400000
  FOOD_PRESERVATION                   2.792308
  MECHANICAL_ENERGY_COMM              1.212121
  MECHANICAL_ENERGY_IND               1.111111
  MECHANICAL_ENERGY_MOV_AGR           1.111111
  MECHANICAL_ENERGY_FIX_AGR           1.111111
  MECHANICAL_ENERGY_MIN               1.111111
  MECHANICAL_ENERGY_FISH_OTHERS       1.111111
  NON_ENERGY                          1.000000


## 2. Helper — empty Demands table

Each cluster gets a fresh copy of this 21-row table (one row per end-use layer). All sector columns start at zero and are filled in by the three sources below.

In [4]:
def create_empty_demands():
    columns = [
        "Category", "Subcategory", "parameter name",
        "HOUSEHOLDS", "SERVICES", "INDUSTRY", "TRANSPORTATION",
        "PUBLIC_LIGHTING", "AGRICULTURE", "MINING", "FISHING_OTHERS",
        "Units",
    ]
    rows = [
        ["Electricity", "Electricity",                            "ELECTRICITY",                   "[GWh]"],
        ["Lighting",    "Building lighting",                      "LIGHTING_R_C",                  "[GWh]"],
        ["Lighting",    "Public lighting",                        "LIGHTING_P",                    "[GWh]"],
        ["Heat",        "High temperature",                       "HEAT_HIGH_T",                   "[GWh]"],
        ["Heat",        "Space heating",                          "HEAT_LOW_T_SH",                 "[GWh]"],
        ["Heat",        "Hot water",                              "HEAT_LOW_T_HW",                 "[GWh]"],
        ["Heat",        "Cooking",                                "COOKING",                       "[GWh]"],
        ["Cold",        "Process cooling",                        "PROCESS_COOLING",               "[GWh]"],
        ["Cold",        "Space cooling",                          "SPACE_COOLING",                 "[GWh]"],
        ["Cold",        "Food preservation",                      "FOOD_PRESERVATION",             "[GWh]"],
        ["Mobility",    "Passenger",                              "MOBILITY_PASSENGER",            "[Mpkm]"],
        ["Mobility",    "Freight",                                "MOBILITY_FREIGHT",              "[Mtkm]"],
        ["Mobility",    "Long-haul passenger flights",            "AVIATION_LONG_HAUL",            "[Mpkm]"],
        ["Mobility",    "International shipping",                 "SHIPPING",                      "[Mtkm]"],
        ["Mechanical",  "Mechanical energy commercial",           "MECHANICAL_ENERGY_COMM",        "[GWh]"],
        ["Mechanical",  "Mechanical energy industrial",           "MECHANICAL_ENERGY_IND",         "[GWh]"],
        ["Mechanical",  "Mechanical energy agriculture mobility", "MECHANICAL_ENERGY_MOV_AGR",     "[GWh]"],
        ["Mechanical",  "Mechanical energy agriculture fixed",    "MECHANICAL_ENERGY_FIX_AGR",     "[GWh]"],
        ["Mechanical",  "Mechanical energy mining",               "MECHANICAL_ENERGY_MIN",         "[GWh]"],
        ["Mechanical",  "Mechanical energy fishing",              "MECHANICAL_ENERGY_FISH_OTHERS", "[GWh]"],
        ["Non-energy",  "Non-energy",                             "NON_ENERGY",                    "[GWh]"],
    ]
    df = pd.DataFrame(columns=columns)
    for i, row in enumerate(rows):
        df.loc[i] = [row[0], row[1], row[2], 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, row[3]]
    return df

## 3. Source A — Grid-connected demand (AETN measurements)

AETN (the regional utility) measures total electricity sold per municipality. We have decomposed those totals into sectors and end-uses in a separate notebook (`source_A_all_sectors_end_uses.csv`).

Values are in **MWh of final electrical energy** → converted to GWh of useful service by dividing by the `|ELECTRICITY|` coefficient of each end-use layer.

`SERVICES_OTHER` is merged into `SERVICES` (no EnergyScope distinction).

In [5]:
sa = pd.read_csv("../../exctraction of data/output/source_A_all_sectors_end_uses.csv")
sa["muni_ramp"] = sa["municipality"].map(SOURCE_A_TO_RAMP)

unmapped = sa[sa["muni_ramp"].isna()]["municipality"].unique()
if len(unmapped) > 0:
    raise ValueError(f"Unmapped Source A municipalities: {unmapped}")

sa["sector"] = sa["sector"].replace("SERVICES_OTHER", "SERVICES")
sa["cluster"] = sa["muni_ramp"].map(MUNI_TO_CLUSTER)

SECTOR_COLS = ["HOUSEHOLDS", "SERVICES", "INDUSTRY", "TRANSPORTATION",
               "PUBLIC_LIGHTING", "AGRICULTURE", "MINING", "FISHING_OTHERS"]

source_a_by_cluster = {}
print("Source A — grid demand by cluster (MWh final → GWh useful):")
for cluster_id in sorted(CLUSTERS):
    sa_c = sa[sa["cluster"] == cluster_id]
    result = {}
    for (sector, end_use), group in sa_c.groupby(["sector", "end_use"]):
        gwh_final = group["MWh"].sum() / 1000.0
        gwh_useful = gwh_final / LAYER_TO_COEFF.get(end_use, 1.0)
        result[(sector, end_use)] = gwh_useful
    source_a_by_cluster[cluster_id] = result
    total = sum(result.values())
    print(f"  C{cluster_id}: {total:.4f} GWh useful  (from {sa_c['MWh'].sum()/1000:.4f} GWh final)")

Source A — grid demand by cluster (MWh final → GWh useful):
  C1: 11.4018 GWh useful  (from 11.7483 GWh final)
  C2: 0.2721 GWh useful  (from 0.2427 GWh final)
  C3: 90.6238 GWh useful  (from 90.0889 GWh final)
  C4: 23.6977 GWh useful  (from 22.8816 GWh final)
  C5: 55.3653 GWh useful  (from 55.1415 GWh final)


## 4. Source B — Off-grid RAMP reality simulation

RAMP was run for the 9,325 off-grid households across the 21 municipalities. The annual summary CSV gives one row per municipality with one column per appliance category, in **GWh of electrical energy**.

Same conversion as Source A: divide by the `|ELECTRICITY|` coefficient to get GWh of useful service.

In [6]:
ramp = pd.read_csv("data ramp/ramp_reality_annual_summary.csv")
ramp = ramp[ramp["municipality"] != "TOTAL"].copy()
ramp["cluster"] = ramp["municipality"].map(MUNI_TO_CLUSTER)

missing = set(MUNI_TO_CLUSTER) - set(ramp["municipality"])
if missing:
    print(f"WARNING: {len(missing)} municipalities missing from RAMP reality (demand set to 0): {missing}")

source_b_by_cluster = {}
print("Source B — off-grid RAMP demand by cluster (GWh elec → GWh useful):")
for cluster_id in sorted(CLUSTERS):
    ramp_c = ramp[ramp["cluster"] == cluster_id]
    result = {}
    for _, row in ramp_c.iterrows():
        for col, (sector, end_use) in MAPPING_RAMP_REALITY.items():
            if col in row.index and not pd.isna(row[col]):
                gwh_useful = float(row[col]) / LAYER_TO_COEFF.get(end_use, 1.0)
                key = (sector, end_use)
                result[key] = result.get(key, 0.0) + gwh_useful
    source_b_by_cluster[cluster_id] = result
    print(f"  C{cluster_id}: {sum(result.values()):.4f} GWh useful")

Source B — off-grid RAMP demand by cluster (GWh elec → GWh useful):
  C1: 0.5493 GWh useful
  C2: 0.0734 GWh useful
  C3: 0.6165 GWh useful
  C4: 0.8651 GWh useful
  C5: 0.0927 GWh useful


## 5. Cooking demand — Census 2024

Traditional cooking (wood, LPG, ...) is **non-electric** and therefore not captured by RAMP or AETN measurements. We estimate it from the census using a fuel-independent useful energy intensity.

**Households counted:** `total_2024 − no_cocina_2024 − electric_cookers_2024`  
Electric cooking households are excluded — their cooking energy already appears in Source A and would be double-counted.

**Useful energy intensity:** 1,344 kWh/household/year — expert estimate from Pablo Jimenez Zabalaga (Roger Arias thesis, 2024–2025).  
EnergyScope's optimizer then picks the fuel mix (STOVE_WOOD, STOVE_LPG, STOVE_ELEC) to meet that useful demand.

This demand goes into **HOUSEHOLDS × COOKING** only. SERVICES × COOKING (restaurant kitchens) is already in Source B.

In [7]:
USEFUL_COOKING_PER_HH_GWh = 0.001344023  # GWh/household/year

# Column indices (0-based) in CSV_final.csv — 2024 cooking section
COL_DEPT       = 1
COL_MUNI       = 3
COL_COOK_TOTAL = 46  # 2024 | Total
COL_COOK_ELEC  = 52  # 2024 | Electricidad
COL_NO_COCINA  = 54  # 2024 | No cocina

# header=None keeps integer column positions consistent with the original Excel layout
csv_final = pd.read_csv(
    "../../exctraction of data/output/CSV_final.csv",
    header=None
)

cooking_data = {}  # RAMP municipality name → non-electric cooking households
for i in range(1, csv_final.shape[0]):  # row 0 is the header; province rows are caught by pd.isna below
    name = csv_final.iloc[i, COL_MUNI]
    dept = csv_final.iloc[i, COL_DEPT]
    if pd.isna(name):
        continue
    key = (str(name).strip(), str(dept).strip())
    if key not in CSVFINAL_TO_RAMP:
        continue
    ramp_name = CSVFINAL_TO_RAMP[key]
    def _parse_int_cell(x):
        if pd.isna(x):
            return 0
        s = str(x).strip().replace('\xa0', '').replace(' ', '')
        if s == '' or s in ('-', 'nan'):
            return 0
        try:
            return int(float(s))
        except Exception:
            try:
                return int(s)
            except Exception:
                return 0

    total     = _parse_int_cell(csv_final.iloc[i, COL_COOK_TOTAL])
    elec      = _parse_int_cell(csv_final.iloc[i, COL_COOK_ELEC])
    no_cocina = _parse_int_cell(csv_final.iloc[i, COL_NO_COCINA])
    cooking_data[ramp_name] = total - no_cocina - elec

total_hh = sum(cooking_data.values())
print(f"Non-electric cooking households: {total_hh} (expected 82,328 = 82,501 − 173 electric)")
assert total_hh == 82328, f"Mismatch: got {total_hh}"

cooking_by_cluster = {}
for cluster_id in sorted(CLUSTERS):
    hh  = sum(cooking_data.get(m, 0) for m in CLUSTERS[cluster_id])
    gwh = hh * USEFUL_COOKING_PER_HH_GWh
    cooking_by_cluster[cluster_id] = gwh
    print(f"  C{cluster_id}: {hh:>6d} HH → {gwh:.4f} GWh useful")

Non-electric cooking households: 82328 (expected 82,328 = 82,501 − 173 electric)
  C1:  10698 HH → 14.3784 GWh useful
  C2:    794 HH → 1.0672 GWh useful
  C3:  39489 HH → 53.0741 GWh useful
  C4:  16343 HH → 21.9654 GWh useful
  C5:  15004 HH → 20.1657 GWh useful


## 6. Build Demands.csv per cluster

For each cluster:
1. Start from an empty demand table
2. Add Source A (grid, by sector and end-use)
3. Add Source B (off-grid RAMP, by sector and end-use)
4. Add cooking useful energy (HOUSEHOLDS × COOKING)
5. Save to `output_energyscope/C{k}/Demands.csv`

In [8]:
for cluster_id in sorted(CLUSTERS):
    df = create_empty_demands()

    for (sector, end_use), gwh in source_a_by_cluster.get(cluster_id, {}).items():
        if end_use in MOBILITY_LAYERS or sector not in SECTOR_COLS:
            continue
        df.loc[df["parameter name"] == end_use, sector] += gwh

    for (sector, end_use), gwh in source_b_by_cluster.get(cluster_id, {}).items():
        if end_use in MOBILITY_LAYERS or sector not in SECTOR_COLS:
            continue
        df.loc[df["parameter name"] == end_use, sector] += gwh

    df.loc[df["parameter name"] == "COOKING", "HOUSEHOLDS"] += cooking_by_cluster[cluster_id]

    out_dir  = f"{OUTPUT_DIR}/C{cluster_id}"
    out_path = f"{out_dir}/Demands.csv"
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)

    print(f"Cluster {cluster_id} ({len(CLUSTERS[cluster_id])} municipalities):")
    total = 0.0
    for sec in SECTOR_COLS:
        val = df[sec].astype(float).sum()
        if val > 0:
            print(f"  {sec:<20}: {val:.4f} GWh")
            total += val
    print(f"  {'TOTAL':<20}: {total:.4f} GWh")
    print(f"  → {out_path}")
    print("-" * 50)

Cluster 1 (4 municipalities):
  HOUSEHOLDS          : 22.2566 GWh
  SERVICES            : 3.0354 GWh
  INDUSTRY            : 0.6923 GWh
  PUBLIC_LIGHTING     : 0.3451 GWh
  TOTAL               : 26.3295 GWh
  → output_energyscope/C1/Demands.csv
--------------------------------------------------


Cluster 2 (1 municipalities):
  HOUSEHOLDS          : 1.2730 GWh
  SERVICES            : 0.1182 GWh
  INDUSTRY            : 0.0182 GWh
  PUBLIC_LIGHTING     : 0.0033 GWh
  TOTAL               : 1.4126 GWh
  → output_energyscope/C2/Demands.csv
--------------------------------------------------


Cluster 3 (3 municipalities):
  HOUSEHOLDS          : 106.0352 GWh
  SERVICES            : 29.9512 GWh
  INDUSTRY            : 6.5073 GWh
  PUBLIC_LIGHTING     : 1.8207 GWh
  TOTAL               : 144.3144 GWh
  → output_energyscope/C3/Demands.csv
--------------------------------------------------
Cluster 4 (12 municipalities):
  HOUSEHOLDS          : 36.3672 GWh
  SERVICES            : 8.3416 GWh
  INDUSTRY            : 1.4378 GWh
  PUBLIC_LIGHTING     : 0.3816 GWh
  TOTAL               : 46.5282 GWh
  → output_energyscope/C4/Demands.csv
--------------------------------------------------


Cluster 5 (1 municipalities):
  HOUSEHOLDS          : 50.5250 GWh
  SERVICES            : 20.9778 GWh
  INDUSTRY            : 3.3702 GWh
  PUBLIC_LIGHTING     : 0.7506 GWh
  TOTAL               : 75.6237 GWh
  → output_energyscope/C5/Demands.csv
--------------------------------------------------


## 7. Quick sanity check — Reality vs Sufficiency

If the sufficiency `Demands.csv` files already exist in the same output folder, this cell compares total demand per cluster. Reality demand should be higher than sufficiency (real measured + off-grid), but in the same ballpark.

In [9]:
print("Reality vs Sufficiency — total demand per cluster:")
print(f"{'Cluster':<10} {'Reality':>12} {'Sufficiency':>14} {'Ratio':>8}")
print("-" * 48)

for cluster_id in sorted(CLUSTERS):
    real_path = f"{OUTPUT_DIR}/C{cluster_id}/Demands.csv"
    suff_path = f"{OUTPUT_DIR}/C{cluster_id}/Demands.csv"  # same folder if sufficiency ran first

    df_real = pd.read_csv(real_path, sep=";")
    real_total = sum(df_real[s].astype(float).sum() for s in SECTOR_COLS)

    # Look for a sufficiency output in the sibling folder
    suff_path_alt = f"../sufficiency/output_energyscope/C{cluster_id}/Demands.csv"
    if os.path.exists(suff_path_alt):
        df_suff = pd.read_csv(suff_path_alt, sep=";")
        suff_total = sum(df_suff[s].astype(float).sum() for s in SECTOR_COLS)
        ratio = real_total / suff_total if suff_total > 0 else float('inf')
        print(f"  C{cluster_id:<7}  {real_total:>10.4f}   {suff_total:>12.4f}   {ratio:>6.2f}x")
    else:
        print(f"  C{cluster_id:<7}  {real_total:>10.4f}   {'(not found)':>12}")

Reality vs Sufficiency — total demand per cluster:
Cluster         Reality    Sufficiency    Ratio
------------------------------------------------
  C1           26.3295        31.4472     0.84x
  C2            1.4126         2.3434     0.60x
  C3          144.3144       117.9286     1.22x


  C4           46.5282        48.5021     0.96x
  C5           75.6237        44.9055     1.68x
